# 🏈 Football Lab
### A mission from NFL scouts: do the fastest players really become the best?

Every spring the NFL holds the **Scouting Combine**: future players run the 40-yard dash, jump, and bench press.
Teams spend millions picking players based on these numbers. But **do the fast ones actually play better?**

Grown-up analysts from all over the world are working on this exact question right now in the **NFL Big Data Bowl 2027**.
You're going to answer it yourself, in Python, with real NFL data: **almost 9,000 players since 2000.**

**How it works:**
- Gray boxes are code. Click a box and press **Shift + Enter** to run it.
- Boxes marked 🏈 **Your turn** are where you change or add code yourself.
- Broke something? No big deal, that's how every programmer learns. Read the red message and try again.

---
## Mission 1. Warm-up 🔥
Python remembers things in **variables**. A variable is like a locker with a name tag: you put a value inside.

In [ ]:
player = "Justin Jefferson"
team = "Vikings"
number = 18

print(player, "plays for the", team, "and wears number", number)

### 🏈 Your turn
Change the values to your favorite player and run it again.

In [ ]:
player = "Patrick Mahomes"   # change to your favorite player
team = "Chiefs"              # his team
number = 15                  # his number

print(player, "plays for the", team, "and wears number", number)

Python is also a calculator. Touchdown = 6 points, extra point = 1, field goal = 3.

In [ ]:
touchdowns = 3
extra_points = 3
field_goals = 2

score = touchdowns * 6 + extra_points * 1 + field_goals * 3
print("Final score:", score)

### 🏈 Your turn
Think of a game you watched. Change the numbers below to that team's touchdowns, extra points,
field goals and safeties (a safety is 2 points). Does the score match?

In [ ]:
touchdowns = 3     # change these numbers
extra_points = 3
field_goals = 2
safeties = 0

score = touchdowns * 6 + extra_points * 1 + field_goals * 3 + safeties * 2
print("Final score:", score)

---
## Mission 2. Opening the secret NFL archive 📂
Now we'll load **every Combine result since 2000**. For that we need a library called `pandas` —
it's like a spreadsheet inside Python. The data comes from the open **nflverse** project, which real analysts use.

In [ ]:
import pandas as pd

combine = pd.read_csv("https://github.com/nflverse/nflverse-data/releases/download/combine/combine.csv")

print("Players in the archive:", len(combine))
combine.head()

What the columns mean:

| column | what it is |
|---|---|
| `player_name` | player's name |
| `pos` | position (WR, RB, QB, CB…) |
| `ht`, `wt` | height (feet-inches) and weight (pounds) |
| `forty` | 40-yard dash time in seconds — **lower is faster** |
| `bench` | how many times he bench-pressed 225 pounds |
| `vertical` | vertical jump in inches |
| `draft_round` | which draft round he was picked in (empty = not drafted at all!) |

### 🏈 Your turn
Find your player in the archive. Type the name **exactly**.
(Quarterbacks often skip running at the Combine, so `forty` might be empty for some players: `NaN`.)

In [ ]:
name = "Justin Jefferson"   # change this

combine[combine["player_name"] == name]

---
## Mission 3. Who's the fastest ever? ⚡
Let's sort every player by 40-yard time, from fastest to slowest.

In [ ]:
fastest = combine.dropna(subset=["forty"]).sort_values("forty")
fastest[["player_name", "pos", "forty", "season"]].head(10)

### 🏈 Your turn
1. Guess before you run it: who is the **slowest**? Which position, probably?
2. Change `.head(10)` to `.tail(10)` to see the end of the list. Did you guess right?

In [ ]:
fastest[["player_name", "pos", "forty", "season"]].head(10)   # change head to tail

Now just one position. The fastest wide receivers (WR):

In [ ]:
position = "WR"   # try RB, CB, QB, TE

one_position = combine[combine["pos"] == position]
one_position.sort_values("forty")[["player_name", "forty", "season"]].head(10)

---
## Mission 4. Speedometer 🏎️
If you know the 40-yard time, you can work out the speed in miles per hour.
We'll write our own **function** — a little machine: you put a time in, and a speed comes out.

In [ ]:
def speed_mph(seconds):
    yards_per_second = 40 / seconds
    return round(yards_per_second * 3600 / 1760, 2)

print("Xavier Worthy (4.21 sec):", speed_mph(4.21), "mph")

### 🏈 Your turn: a real experiment!
Go outside with mom or dad and a stopwatch. Measure **40 yards (about 50 big steps)**.
Run it, time yourself, and type in your result. How much slower are you than Xavier Worthy?

In [ ]:
my_time = 8.5   # type your time in seconds

print("My speed:", speed_mph(my_time), "mph")
print("Worthy is faster by", round(my_time - 4.21, 2), "seconds")

**Bonus:** add a new speed column for **every** player at once, in one line!

In [ ]:
combine["speed"] = combine["forty"].apply(speed_mph)
combine.sort_values("speed", ascending=False)[["player_name", "pos", "forty", "speed"]].head(5)

---
## Mission 5. THE BIG QUESTION 🔍
### Do fast receivers become the best ones?

**Guess first.** Write down what you think before you see the answer. Scientists do this too, so nobody can say "I knew it all along" later.

In [ ]:
my_guess = "Yes, the fastest ones get the most yards"   # write what you think

Now let's load stats from **every NFL game from 1999 to 2024** (the 2025–2026 seasons aren't in this file yet)
and add up how many receiving yards each player got in his whole career. (The file is big, it takes about 20 seconds.)

In [ ]:
stats = pd.read_csv(
    "https://github.com/nflverse/nflverse-data/releases/download/player_stats/player_stats.csv",
    low_memory=False,
)
regular = stats[stats["season_type"] == "REG"]
career_yards = regular.groupby("player_display_name")["receiving_yards"].sum()

career_yards.sort_values(ascending=False).head(10)

Let's join the two tables: Combine speed + career yards.
(Players who share a name, like the two Steve Smiths, get removed so we don't mix them up.)

In [ ]:
wr = combine[(combine["pos"] == "WR") & combine["forty"].notna()]
wr = wr.drop_duplicates("player_name", keep=False)

data = wr.merge(career_yards.rename("yards"), left_on="player_name", right_index=True)
print("Receivers with both numbers:", len(data))
data[["player_name", "forty", "yards"]].sort_values("yards", ascending=False).head(10)

Now a **chart**. Every dot is one receiver.
- **Left = faster** (fewer seconds), **right = slower** (more seconds).
- **Up = more yards** in his career.

If fast players really are better, the dots on the left should be the highest.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.scatter(data["forty"], data["yards"], alpha=0.5)

for _, row in data.sort_values("yards", ascending=False).head(8).iterrows():
    plt.annotate(row["player_name"], (row["forty"], row["yards"]), fontsize=8)

plt.xlabel("40-yard dash in seconds   (left = faster, right = slower)")
plt.ylabel("Career receiving yards")
plt.title("Does speed make a great receiver?")
plt.show()

### 🏈 Your turn: be the scout
Look at the chart and answer (you can type right here — double-click this box):
1. Are the fastest players (on the left) the highest?
2. Find the stars who were **slow**. Who are they? Why did they still become great?
3. Was your guess right?

Here are the slow stars: 4.55 or slower, but more than 8,000 yards:

In [ ]:
data[(data["forty"] >= 4.55) & (data["yards"] > 8000)][["player_name", "forty", "yards"]]

And here are fast players who never broke out: 4.35 or faster, but fewer than 1,000 career yards.
We only count players who entered the NFL before 2019 — rookies haven't had time to pile up yards yet.

In [ ]:
data[(data["forty"] <= 4.35) & (data["yards"] < 1000) & (data["season"] < 2019)][["player_name", "forty", "yards"]]

---
## Bonus mission 6. What about running backs? 🧪
Only if you still have energy! Same question, but for running backs (RB) and their **rushing** yards.
Just run this box, nothing to change:

In [ ]:
position = "RB"
yards_type = "rushing_yards"

players = combine[(combine["pos"] == position) & combine["forty"].notna()]
players = players.drop_duplicates("player_name", keep=False)
career = regular.groupby("player_display_name")[yards_type].sum()
both = players.merge(career.rename("yards"), left_on="player_name", right_index=True)

plt.figure(figsize=(10, 6))
plt.scatter(both["forty"], both["yards"], alpha=0.5)
for _, row in both.sort_values("yards", ascending=False).head(8).iterrows():
    plt.annotate(row["player_name"], (row["forty"], row["yards"]), fontsize=8)
plt.xlabel("40-yard dash in seconds   (left = faster, right = slower)")
plt.ylabel(yards_type)
plt.title(position + ": does speed matter?")
plt.show()

### 🏈 Your turn
Now tight ends (TE). In the box below, change **only the first two lines**:
- `"RB"` → `"TE"`
- `"rushing_yards"` → `"receiving_yards"`

Keep the quotes! Then press Shift + Enter.

In [ ]:
position = "RB"               # change to "TE"
yards_type = "rushing_yards"  # change to "receiving_yards"

players = combine[(combine["pos"] == position) & combine["forty"].notna()]
players = players.drop_duplicates("player_name", keep=False)
career = regular.groupby("player_display_name")[yards_type].sum()
both = players.merge(career.rename("yards"), left_on="player_name", right_index=True)

plt.figure(figsize=(10, 6))
plt.scatter(both["forty"], both["yards"], alpha=0.5)
for _, row in both.sort_values("yards", ascending=False).head(8).iterrows():
    plt.annotate(row["player_name"], (row["forty"], row["yards"]), fontsize=8)
plt.xlabel("40-yard dash in seconds   (left = faster, right = slower)")
plt.ylabel(yards_type)
plt.title(position + ": does speed matter?")
plt.show()

---
### 🏆 You finished Football Lab!
You just did what NFL team analysts do: load data, ask a question,
build a chart, and find the answer. Show your chart to someone and explain what you found.